# AI Lung Abnormality Detection

**CheXpert + DenseNet121 — multi-label chest X-ray classification**

End-to-end pipeline: dataset loading → preprocessing → model architecture → training/fine-tuning → evaluation → Grad-CAM explainability → inference/report generation → Hugging Face deployment.

Built for execution on the AI server (dual RTX 4000 Ada, 20GB each). Run phase by phase, top to bottom, in a single kernel session so variables (`train_df`, `model`, `base_model`, etc.) carry over between phases.

# Phase 1 – Environment Setup & Dataset Loading

---

## Cell 1 – Install Required Libraries (Skip if already installed)

In [ ]:
!pip install -q kagglehub

---

## Cell 2 – Import Libraries

In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf

warnings.filterwarnings("ignore")

---

## Cell 3 – Check TensorFlow Version & GPU

In [ ]:
print("=" * 50)
print("TensorFlow Version :", tf.__version__)
print("=" * 50)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print(f"GPU Detected : {gpus[0].name}")
else:
    print("Running on CPU")

---

## Cell 4 – Set Random Seed

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random Seed Set :", SEED)

---

## Cell 5 – Download Dataset & Resolve Path

We use `kagglehub` to download the dataset and resolve its cache path dynamically.
This avoids hardcoding a path like `/root/.cache/...`, which only exists if you're
running as `root` — on the AI server we're running as `administrator`, so the
cache lives under `/home/administrator/.cache/kagglehub/...` instead.

In [ ]:
import kagglehub

DATASET_PATH = kagglehub.dataset_download("ashery/chexpert")

print("Dataset downloaded to:", DATASET_PATH)

---

## Cell 6 – Verify Dataset Structure

In [ ]:
print("Dataset Contents:\n")

for item in os.listdir(DATASET_PATH):
    print(item)

Expected output:

```
train/
valid/
train.csv
valid.csv
```

---

## Cell 7 – Load CSV Files

In [ ]:
train_df = pd.read_csv(
    os.path.join(DATASET_PATH, "train.csv")
)

valid_df = pd.read_csv(
    os.path.join(DATASET_PATH, "valid.csv")
)

print("Train Shape :", train_df.shape)
print("Validation Shape :", valid_df.shape)

---

## Cell 8 – Define Disease Labels

In [ ]:
LABELS = [

    "No Finding",

    "Enlarged Cardiomediastinum",

    "Cardiomegaly",

    "Lung Opacity",

    "Lung Lesion",

    "Edema",

    "Consolidation",

    "Pneumonia",

    "Atelectasis",

    "Pneumothorax",

    "Pleural Effusion",

    "Pleural Other",

    "Fracture",

    "Support Devices"

]

---

## Cell 9 – Clean Labels

Replace missing values and uncertain labels.

In [ ]:
train_df[LABELS] = (
    train_df[LABELS]
    .fillna(0)
    .replace(-1, 0)
)

valid_df[LABELS] = (
    valid_df[LABELS]
    .fillna(0)
    .replace(-1, 0)
)

---

## Cell 10 – Keep Only Frontal X-rays

In [ ]:
train_df = train_df[
    train_df["Frontal/Lateral"] == "Frontal"
].reset_index(drop=True)

valid_df = valid_df[
    valid_df["Frontal/Lateral"] == "Frontal"
].reset_index(drop=True)

print("Frontal Training Images :", len(train_df))
print("Frontal Validation Images :", len(valid_df))

---

## Cell 11 – Fix Kaggle Mirror Paths

The CSV contains:

```
CheXpert-v1.0-small/train/...
```

Your downloaded dataset contains:

```
train/...
```

So we fix the paths.

In [ ]:
train_df["Path"] = train_df["Path"].str.replace(
    "CheXpert-v1.0-small/",
    "",
    regex=False
)

valid_df["Path"] = valid_df["Path"].str.replace(
    "CheXpert-v1.0-small/",
    "",
    regex=False
)

---

## Cell 12 – Create Absolute Image Paths

In [ ]:
train_df["image_path"] = train_df["Path"].apply(
    lambda x: os.path.join(DATASET_PATH, x)
)

valid_df["image_path"] = valid_df["Path"].apply(
    lambda x: os.path.join(DATASET_PATH, x)
)

---

## Cell 13 – Verify Image Exists

In [ ]:
sample_path = train_df["image_path"].iloc[0]

print(sample_path)

print()

print("Exists :", os.path.exists(sample_path))

Expected:

```
Exists : True
```

---

## Cell 14 – Display Sample Image

In [ ]:
img = Image.open(sample_path)

plt.figure(figsize=(6,6))
plt.imshow(img, cmap="gray")
plt.axis("off")
plt.title("Sample Chest X-ray")
plt.show()

---

## Cell 15 – Dataset Summary

In [ ]:
print("=" * 50)

print("Training Images :", len(train_df))

print("Validation Images :", len(valid_df))

print("=" * 50)

---

## Cell 16 – Missing Values Check

In [ ]:
train_df[LABELS].isnull().sum()

Expected:

```
All zeros
```

---

## Cell 17 – Disease Distribution

In [ ]:
disease_counts = train_df[LABELS].sum().sort_values(
    ascending=False
)

plt.figure(figsize=(12,6))

disease_counts.plot(kind="bar")

plt.title("Disease Distribution")

plt.ylabel("Number of Images")

plt.xticks(rotation=60)

plt.show()

---

## Cell 18 – Display Random Images

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12,8))

samples = train_df.sample(6, random_state=42)

for ax, (_, row) in zip(axes.flatten(), samples.iterrows()):

    image = Image.open(row["image_path"])

    ax.imshow(image, cmap="gray")

    ax.axis("off")

plt.tight_layout()

plt.show()

---

## Cell 19 – Final Verification

In [ ]:
print("Dataset Ready!")

print()

print("Training Samples :", len(train_df))

print("Validation Samples :", len(valid_df))

print()

print("Number of Classes :", len(LABELS))

Expected output:

```
Dataset Ready!

Training Samples : XXXXX

Validation Samples : XXXXX

Number of Classes : 14
```

---

# ✅ Phase 1 Complete

At the end of this phase, you will have:

* TensorFlow configured with GPU detection.
* CheXpert CSVs loaded.
* Kaggle mirror paths corrected.
* Only frontal chest X-rays selected.
* Labels cleaned (`NaN` and `-1` handled).
* Absolute image paths created and verified.
* Dataset statistics and disease distribution visualized.
* Sample X-ray images displayed.

The notebook is now ready for **Phase 2**, where we'll build the `tf.data` pipeline with preprocessing, augmentation, batching, and prefetching for efficient training.

# Phase 2 – Data Preprocessing & TensorFlow Pipeline

---

## Cell 1 – Configuration

In [ ]:
# ======================================================
# Configuration
# ======================================================

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

print(f"Image Size : {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch Size : {BATCH_SIZE}")

---

## Cell 2 – Convert Labels to NumPy Arrays

In [ ]:
# ======================================================
# Prepare Labels
# ======================================================

train_labels = train_df[LABELS].astype(np.float32).values

valid_labels = valid_df[LABELS].astype(np.float32).values

print(train_labels.shape)
print(valid_labels.shape)

Expected Output

```
(XXXXX, 14)
(XXXXX, 14)
```

---

## Cell 3 – Data Augmentation

In [ ]:
# ======================================================
# Data Augmentation
# ======================================================

data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.05),

    tf.keras.layers.RandomZoom(0.10),

    tf.keras.layers.RandomContrast(0.10),

], name="augmentation")

---

## Cell 4 – Image Loading Function

In [ ]:
# ======================================================
# Read Image
# ======================================================

def load_image(path, labels):

    image = tf.io.read_file(path)

    image = tf.image.decode_jpeg(image, channels=3)

    image = tf.image.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    image = tf.cast(image, tf.float32)

    image = image / 255.0

    return image, labels

---

## Cell 5 – Training Preprocessing

In [ ]:
# ======================================================
# Training Pipeline
# ======================================================

def preprocess_train(image, labels):

    image = data_augmentation(image)

    return image, labels

---

## Cell 6 – Validation Preprocessing

In [ ]:
# ======================================================
# Validation Pipeline
# ======================================================

def preprocess_valid(image, labels):

    return image, labels

---

## Cell 7 – Create TensorFlow Dataset

In [ ]:
# ======================================================
# Create Dataset
# ======================================================

train_ds = tf.data.Dataset.from_tensor_slices(

    (
        train_df["image_path"].values,
        train_labels
    )

)

valid_ds = tf.data.Dataset.from_tensor_slices(

    (
        valid_df["image_path"].values,
        valid_labels
    )

)

print(train_ds)
print(valid_ds)

---

## Cell 8 – Apply Loading Function

In [ ]:
# ======================================================
# Read Images
# ======================================================

train_ds = train_ds.map(

    load_image,

    num_parallel_calls=AUTOTUNE

)

valid_ds = valid_ds.map(

    load_image,

    num_parallel_calls=AUTOTUNE

)

---

## Cell 9 – Apply Data Augmentation

In [ ]:
# ======================================================
# Apply Augmentation
# ======================================================

train_ds = train_ds.map(

    preprocess_train,

    num_parallel_calls=AUTOTUNE

)

valid_ds = valid_ds.map(

    preprocess_valid,

    num_parallel_calls=AUTOTUNE

)

---

## Cell 10 – Shuffle Dataset

**Fixed 2026-07-23 — this was the real cause of today's slow/unstable
retrain attempts.** `buffer_size=len(train_df)` shuffles by holding the
*entire* training set in memory after preprocessing — for 191,027 images at
224×224×3 float32 (~0.57MB each), that's **~110GB**, confirmed by the
retrain kernel process actually using 142GB RSS before it was killed.
That memory pressure is the likely real explanation for the repeated
`InvalidArgumentError: Invalid incarnation id` crashes during multi-GPU
attempts earlier today, on top of just making every shuffle-buffer refill
take 90-100+ seconds. `buffer_size=5000` (a standard size for
image pipelines this large — see TF's own `tf.data` performance guide)
gives effectively equivalent shuffle quality at a tiny fraction of the
memory and refill time.

In [ ]:
# ======================================================
# Shuffle
# ======================================================

train_ds = train_ds.shuffle(

    buffer_size=5000,

    seed=SEED

)

---

## Cell 11 – Batch Dataset

`train_ds.repeat()` matters once Phase 4 trains under `MirroredStrategy`
(2 GPUs): without it, `model.fit()` relies on the dataset naturally running
out (`StopIteration`) to detect the end of each epoch, and that
"exhaustion" path is what triggered `InvalidArgumentError: Invalid
incarnation id` on `MultiDeviceIteratorGetNextFromShard` the first time we
trained multi-GPU — a known TensorFlow/Keras issue with distributed
`tf.data` iterators. Making `train_ds` infinite and telling `model.fit()`
exactly how many steps make up an epoch (Phase 4 Cell 4's
`STEPS_PER_EPOCH`) sidesteps that code path entirely, which is also the
pattern TensorFlow's own multi-GPU guides use. `valid_ds` stays
un-repeated — Phase 5 iterates over it exactly once to build its evaluation
arrays, so it needs to stay finite.

In [ ]:
# ======================================================
# Batch
# ======================================================

train_ds = train_ds.batch(BATCH_SIZE).repeat()

valid_ds = valid_ds.batch(BATCH_SIZE)

---

## Cell 12 – Prefetch

In [ ]:
# ======================================================
# Prefetch
# ======================================================

train_ds = train_ds.prefetch(AUTOTUNE)

valid_ds = valid_ds.prefetch(AUTOTUNE)

---

## Cell 13 – Verify Dataset

In [ ]:
images, labels = next(iter(train_ds))

print("Images Shape :", images.shape)

print("Labels Shape :", labels.shape)

Expected

```
Images Shape : (32,224,224,3)

Labels Shape : (32,14)
```

---

## Cell 14 – Display Sample Images

In [ ]:
plt.figure(figsize=(12,10))

for i in range(9):

    plt.subplot(3,3,i+1)

    plt.imshow(images[i])

    plt.axis("off")

plt.tight_layout()

plt.show()

---

## Cell 15 – Display Labels

In [ ]:
sample = labels[0].numpy()

print("Diseases Present\n")

for disease, value in zip(LABELS, sample):

    if value == 1:

        print("✓", disease)

---

## Cell 16 – Count Disease Frequency

In [ ]:
disease_frequency = train_df[LABELS].sum()

display(disease_frequency.sort_values(ascending=False))

---

## Cell 17 – Disease Distribution Plot

In [ ]:
plt.figure(figsize=(12,6))

disease_frequency.sort_values().plot(

    kind="barh"

)

plt.xlabel("Number of Images")

plt.title("Disease Frequency")

plt.show()

---

## Cell 18 – Dataset Information

In [ ]:
print("="*60)

print("Training Samples :", len(train_df))

print("Validation Samples :", len(valid_df))

print()

print("Batch Size :", BATCH_SIZE)

print()

print("Image Size :", IMG_SIZE)

print()

print("Classes :", len(LABELS))

print("="*60)

---

## Cell 19 – Test Pipeline Speed

In [ ]:
%%time

for images, labels in train_ds.take(5):

    pass

---

## Cell 20 – Final Verification

In [ ]:
print("Phase 2 Completed Successfully!")

print()

print("TensorFlow Dataset Ready")

print("DenseNet121 Training Ready")

Expected Output:

```
Phase 2 Completed Successfully!

TensorFlow Dataset Ready

DenseNet121 Training Ready
```

---

# ✅ Phase 2 Complete

At this stage, your project has:

* ✔️ TensorFlow `tf.data` pipeline
* ✔️ Automatic image loading from disk
* ✔️ Image resizing to **224 × 224**
* ✔️ Pixel normalization (`0–1`)
* ✔️ Data augmentation for training
* ✔️ Efficient shuffling, batching, and prefetching
* ✔️ Batch and label verification
* ✔️ Disease distribution analysis

Your data pipeline is now optimized and ready for **Phase 3**, where you'll build the **DenseNet121** model with a custom multi-label classification head, compile it with appropriate loss/metrics, and prepare it for training.

# Phase 3 – Build DenseNet121 Model

---

## Cell 1 – Import Required Modules

In [ ]:
# ======================================================
# Import Model Components
# ======================================================

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
    BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

---

# Cell 2 – Model Configuration & Mixed Precision

**Multi-GPU update (2026-07-23): attempted and reverted.** Wrapped model
build/compile in `tf.distribute.MirroredStrategy().scope()` to use both
GPUs. Three consecutive retrain attempts on the AI server all failed in
ways specific to the distributed path: `InvalidArgumentError: Invalid
incarnation id` on `MultiDeviceIteratorGetNextFromShard` (crashed at step 2,
then again at step 41 after adding `train_ds.repeat()` +
`steps_per_epoch`), and `loss: nan` reappeared on the second attempt despite
`clipnorm=1.0`, alongside `recall` metric values above 1.0 (mathematically
impossible, pointing at corrupted cross-replica metric aggregation, not
just a data-pipeline timing issue). Reverted to single-GPU here rather than
keep guessing at an unattended multi-GPU fix — single-GPU is what completed
the original 2026-07-22 run cleanly end-to-end. Revisit multi-GPU later
(candidates: newer/older TensorFlow build, or running training as a plain
`.py` script instead of through a Jupyter kernel, since `MultiDeviceIterator`
+ ipykernel is a specific combination with known GitHub issues) — not
blocking on it now. Phase 2's `train_ds.repeat()` + Phase 4's
`STEPS_PER_EPOCH` are harmless on single GPU too, so those stayed.

**Speed update (2026-07-23):** the first clean single-GPU timing we saw
(from the earlier, since-reverted multi-GPU run) was ~328ms/step —
5,970 steps/epoch × 328ms ≈ 33 minutes/epoch, well over the ~10-minute
target. `tf.keras.mixed_precision` runs most ops (the DenseNet121 conv
layers especially) in `bfloat16` on the RTX 4000 Ada's Tensor Cores instead
of `float32`, which is typically a 1.5–3x throughput win on this class of
GPU with minimal accuracy impact — BF16 keeps FP32's exponent range, so
(unlike FP16) it doesn't need loss-scaling to stay numerically stable. Set
here, before the model is built, since layers pick up the global policy at
creation time. Kept deliberately isolated from batch size / learning rate
changes (per the multi-GPU lesson above: bundle fewer risky changes
together, or a regression is hard to attribute) — measure step time after
this alone before deciding if more speed levers are needed.

In [ ]:
# ======================================================
# Model Configuration
# ======================================================

NUM_CLASSES = len(LABELS)

LEARNING_RATE = 1e-4

print("Classes :", NUM_CLASSES)
print("Learning Rate :", LEARNING_RATE)

# ======================================================
# Mixed Precision
# ======================================================

tf.keras.mixed_precision.set_global_policy("mixed_bfloat16")

print("Compute dtype :", tf.keras.mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", tf.keras.mixed_precision.global_policy().variable_dtype)

---

# Cell 3 – Load Pretrained DenseNet121

In [ ]:
# ======================================================
# Load DenseNet121
# ======================================================

base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

print("DenseNet121 Loaded Successfully")

---

# Cell 4 – Freeze Base Model

We first train only the custom classifier.

In [ ]:
# ======================================================
# Freeze DenseNet
# ======================================================

base_model.trainable = False

print("Trainable :", base_model.trainable)

---

# Cell 5 – Build the Classification Head

Under `mixed_bfloat16`, every layer computes in bf16 by default — including
this final one. Keras's own mixed-precision guide recommends keeping the
last layer's output (and therefore the loss computation) in float32 for
numerical stability, so `dtype="float32"` is set explicitly here; it
overrides the global policy for just this layer's output cast, not its
(still bf16) compute.

In [ ]:
# ======================================================
# Build Model
# ======================================================

inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# Augmentation is already applied to train_ds in Phase 2's tf.data
# pipeline (train only, validation skipped) — not repeated here to avoid
# double-augmenting training images. Also, `data_augmentation` was already
# built on unbatched (224, 224, 3) tensors when Phase 2's `.map()` first
# called it, so calling it again here on a batched (None, 224, 224, 3)
# `Input` would raise a shape-mismatch error even if we wanted to reuse it.

# DenseNet Feature Extractor
x = base_model(inputs, training=False)

# Global Pooling
x = GlobalAveragePooling2D()(x)

# Batch Normalization
x = BatchNormalization()(x)

# Fully Connected Layer
x = Dense(
    512,
    activation="relu"
)(x)

# Dropout
x = Dropout(0.4)(x)

# Another Dense Layer
x = Dense(
    256,
    activation="relu"
)(x)

x = Dropout(0.3)(x)

# Output Layer
outputs = Dense(
    NUM_CLASSES,
    activation="sigmoid",
    name="predictions",
    dtype="float32"
)(x)

model = Model(
    inputs,
    outputs,
    name="DenseNet121_CheXpert"
)

---

# Cell 6 – Model Architecture

In [ ]:
model.summary()

Expected Architecture

```text
Input

↓

Data Augmentation

↓

DenseNet121

↓

GlobalAveragePooling2D

↓

BatchNormalization

↓

Dense(512)

↓

Dropout

↓

Dense(256)

↓

Dropout

↓

Dense(14 Sigmoid)
```

---

# Cell 7 – Compile Model

Since this is a **multi-label classification** problem, we use:

* Binary Focal Crossentropy
* Binary Accuracy
* AUC

Plain binary crossentropy treats every label equally, but CheXpert's 14
labels are heavily imbalanced (e.g. Pneumonia is positive in only ~4% of
images). With a flat 0.5 threshold, that let the model minimize loss by
predicting "negative" for every rare class and still score well overall —
10 of 14 labels came back at exactly 0 precision/recall in the first full
training run (see project memory). Binary Focal Crossentropy (Lin et al.,
2017, "Focal Loss for Dense Object Detection") down-weights the loss
contribution of examples the model already classifies confidently — mostly
the abundant, easy negatives — so gradient updates focus more on the hard,
rare positives instead of being drowned out. `gamma=2.0` and `alpha=0.25`
are the paper's standard defaults; `apply_class_balancing=True` turns on
the `alpha` term so it isn't a no-op.

The first attempt at this run (single global batch across 2 mirrored GPUs)
hit `loss: nan` partway through Stage 1 and never recovered — once a NaN
gradient lands, it corrupts the weights permanently for the rest of
training. `Adam(..., clipnorm=1.0)` caps each gradient's norm before the
weight update, which is the standard defensive fix for this regardless of
which exact op produced the bad gradient — cheap insurance that doesn't
change behavior when gradients are already well-scaled.

`jit_compile=True` fuses the training step into an XLA-compiled graph
(same mechanism we already saw fire automatically during earlier runs —
`"Compiled cluster using XLA!"` — made explicit and guaranteed here rather
than left to Keras's default heuristics). `steps_per_execution=8` runs 8
training steps per Python/TF-dispatch round-trip instead of 1, cutting
per-step host-side overhead — this doesn't change what gets computed, only
how many steps get batched into a single traced call before control
returns to Python (so callbacks/logging update every 8 steps instead of
every step, not a training-quality change).

In [ ]:
# ======================================================
# Compile Model
# ======================================================

model.compile(

    optimizer=Adam(
        learning_rate=LEARNING_RATE,
        clipnorm=1.0
    ),

    loss=tf.keras.losses.BinaryFocalCrossentropy(
        apply_class_balancing=True,
        alpha=0.25,
        gamma=2.0,
    ),

    metrics=[

        tf.keras.metrics.BinaryAccuracy(
            name="binary_accuracy"
        ),

        tf.keras.metrics.AUC(
            multi_label=True,
            name="auc"
        ),

        tf.keras.metrics.Precision(
            name="precision"
        ),

        tf.keras.metrics.Recall(
            name="recall"
        )

    ],

    jit_compile=True,

    steps_per_execution=8

)

---

# Cell 8 – Count Parameters

In [ ]:
# ======================================================
# Parameter Count
# ======================================================

trainable = np.sum(
    [np.prod(v.shape) for v in model.trainable_weights]
)

non_trainable = np.sum(
    [np.prod(v.shape) for v in model.non_trainable_weights]
)

print("Trainable Parameters :", f"{trainable:,}")
print("Frozen Parameters    :", f"{non_trainable:,}")
print("Total Parameters     :", f"{model.count_params():,}")

Example Output

```text
Trainable Parameters : 820,494

Frozen Parameters : 7,037,504

Total Parameters : 7,857,998
```

---

# Cell 9 – Verify Forward Pass

In [ ]:
# ======================================================
# Test Forward Pass
# ======================================================

images, labels = next(iter(train_ds))

predictions = model(images)

print("Prediction Shape :", predictions.shape)

Expected

```text
Prediction Shape : (32,14)
```

---

# Cell 10 – Inspect Predictions

In [ ]:
# ======================================================
# Sample Predictions
# ======================================================

sample_prediction = predictions[0].numpy()

prediction_df = pd.DataFrame({

    "Disease": LABELS,

    "Probability": sample_prediction

})

prediction_df = prediction_df.sort_values(
    by="Probability",
    ascending=False
)

prediction_df

Initially, the probabilities will be close to **0.5** because the model hasn't been trained yet.

---

# Cell 11 – Save Model Architecture

In [ ]:
# ======================================================
# Save Initial Architecture
# ======================================================

model.save("initial_densenet121.keras")

print("Initial model saved successfully.")

---

# Cell 12 – Final Verification

In [ ]:
print("="*60)
print("Phase 3 Completed Successfully!")
print("="*60)

print()

print("Model Name :", model.name)

print()

print("Classes :", NUM_CLASSES)

print()

print("Loss Function : Binary Focal Crossentropy")

print()

print("Activation : Sigmoid")

print()

print("Ready for Training!")

Expected Output

```text
============================================================
Phase 3 Completed Successfully!
============================================================

Model Name : DenseNet121_CheXpert

Classes : 14

Loss Function : Binary Focal Crossentropy

Activation : Sigmoid

Ready for Training!
```

---

# ✅ Phase 3 Complete

At the end of this phase, you have:

* ✔️ Pretrained **DenseNet121** (ImageNet weights)
* ✔️ Frozen feature extractor
* ✔️ Custom classification head (512 → 256 → 14)
* ✔️ Sigmoid outputs for multi-label prediction
* ✔️ Binary Focal Crossentropy loss (handles class imbalance across the 14 labels)
* ✔️ AUC, Precision, Recall, Binary Accuracy metrics
* ✔️ Initial model saved

---

## 🚀 Next: Phase 4 – Training & Fine-Tuning

We'll implement a robust training strategy including:

* EarlyStopping
* ModelCheckpoint
* ReduceLROnPlateau
* Initial training with the frozen backbone
* Fine-tuning the last DenseNet blocks
* Saving the best-performing model
* Training history plots (Loss, AUC, Precision, Recall)

This will produce a model ready for evaluation and Grad-CAM visualization.

# Phase 4 – Training & Fine-Tuning

---

## Cell 1 – Import Callbacks

In [ ]:
# ======================================================
# Import Callbacks
# ======================================================

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
    CSVLogger
)

---

# Cell 2 – Create Model Save Directory

In [ ]:
# ======================================================
# Create Directory
# ======================================================

MODEL_DIR = "saved_models"

os.makedirs(MODEL_DIR, exist_ok=True)

print("Model Directory:", MODEL_DIR)

---

# Cell 3 – Define Callbacks

In [ ]:
# ======================================================
# Callbacks
# ======================================================

early_stopping = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=os.path.join(
        MODEL_DIR,
        "best_densenet121.keras"
    ),
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    verbose=1
)

csv_logger = CSVLogger(
    os.path.join(
        MODEL_DIR,
        "training_log.csv"
    )
)

callbacks = [
    early_stopping,
    reduce_lr,
    checkpoint,
    csv_logger
]

---

# Cell 4 – Stage 1 Training Configuration

`train_ds` is now infinite (Phase 2 Cell 11's `.repeat()`), so `model.fit()`
needs to be told explicitly how many steps make up one epoch — this
matches the same batch count each epoch would have covered naturally
before (`ceil(len(train_df) / BATCH_SIZE)`), just computed up front instead
of relying on the dataset running out.

In [ ]:
# ======================================================
# Stage 1
# ======================================================

INITIAL_EPOCHS = 10

STEPS_PER_EPOCH = (len(train_df) + BATCH_SIZE - 1) // BATCH_SIZE

print("Stage 1 Training")
print("Frozen DenseNet")
print("Steps per Epoch :", STEPS_PER_EPOCH)

---

# Cell 5 – Train Classification Head

In [ ]:
# ======================================================
# Stage 1 Training
# ======================================================

history_initial = model.fit(

    train_ds,

    validation_data=valid_ds,

    epochs=INITIAL_EPOCHS,

    steps_per_epoch=STEPS_PER_EPOCH,

    callbacks=callbacks,

    verbose=1

)

---

# Cell 6 – Save Stage 1 Model

In [ ]:
# ======================================================
# Save Stage 1
# ======================================================

model.save(

    os.path.join(

        MODEL_DIR,

        "stage1_model.keras"

    )

)

print("Stage 1 Model Saved")

---

# Cell 7 – Unfreeze DenseNet

Instead of training the entire DenseNet, we'll fine-tune only the **last 100 layers**.

In [ ]:
# ======================================================
# Fine Tune
# ======================================================

base_model.trainable = True

print("DenseNet Unfrozen")

---

# Cell 8 – Freeze Early Layers

In [ ]:
# ======================================================
# Freeze Early Layers
# ======================================================

for layer in base_model.layers[:-100]:

    layer.trainable = False

print("Last 100 layers are trainable.")

---

# Cell 9 – Recompile Model

Use a smaller learning rate for fine-tuning. `compile()` resets the loss, so
this must repeat Phase 3 Cell 7's `BinaryFocalCrossentropy` — recompiling
with the string `"binary_crossentropy"` here would silently undo the
class-imbalance fix for the entire fine-tuning stage. Same reasoning
applies to `jit_compile=True` and `steps_per_execution=8` — the mixed
precision *policy* set in Phase 3 Cell 2 stays active for the rest of the
kernel session, but `compile()` itself is a fresh call, so the XLA/step-
batching flags need repeating here too.

In [ ]:
# ======================================================
# Recompile
# ======================================================

model.compile(

    optimizer=Adam(
        learning_rate=1e-5,
        clipnorm=1.0
    ),

    loss=tf.keras.losses.BinaryFocalCrossentropy(
        apply_class_balancing=True,
        alpha=0.25,
        gamma=2.0,
    ),

    metrics=[

        tf.keras.metrics.BinaryAccuracy(
            name="binary_accuracy"
        ),

        tf.keras.metrics.AUC(
            multi_label=True,
            name="auc"
        ),

        tf.keras.metrics.Precision(
            name="precision"
        ),

        tf.keras.metrics.Recall(
            name="recall"
        )

    ],

    jit_compile=True,

    steps_per_execution=8

)

---

# Cell 10 – Fine-Tuning Configuration

In [ ]:
# ======================================================
# Stage 2
# ======================================================

FINE_TUNE_EPOCHS = 15

TOTAL_EPOCHS = INITIAL_EPOCHS + FINE_TUNE_EPOCHS

print("Fine Tuning Started")

---

# Cell 11 – Fine-Tune Model

In [ ]:
# ======================================================
# Fine Tune
# ======================================================

history_fine = model.fit(

    train_ds,

    validation_data=valid_ds,

    epochs=TOTAL_EPOCHS,

    initial_epoch=history_initial.epoch[-1] + 1,

    steps_per_epoch=STEPS_PER_EPOCH,

    callbacks=callbacks,

    verbose=1

)

---

# Cell 12 – Save Final Model

In [ ]:
# ======================================================
# Save Final Model
# ======================================================

model.save(

    os.path.join(

        MODEL_DIR,

        "lung_abnormality_model.keras"

    )

)

print("Final Model Saved Successfully")

---

# Cell 13 – Load Best Model

In [ ]:
# ======================================================
# Load Best Model
# ======================================================

best_model = tf.keras.models.load_model(

    os.path.join(

        MODEL_DIR,

        "best_densenet121.keras"

    )

)

print("Best Model Loaded")

---

# Cell 14 – Plot Training Loss

In [ ]:
# ======================================================
# Training Loss
# ======================================================

plt.figure(figsize=(8,5))

plt.plot(
    history_initial.history["loss"],
    label="Train Loss"
)

plt.plot(
    history_initial.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Stage 1 Loss")

plt.legend()

plt.show()

---

# Cell 15 – Plot AUC

In [ ]:
# ======================================================
# AUC
# ======================================================

plt.figure(figsize=(8,5))

plt.plot(
    history_initial.history["auc"],
    label="Train AUC"
)

plt.plot(
    history_initial.history["val_auc"],
    label="Validation AUC"
)

plt.xlabel("Epoch")

plt.ylabel("AUC")

plt.title("Stage 1 AUC")

plt.legend()

plt.show()

---

# Cell 16 – Evaluate Best Model

In [ ]:
# ======================================================
# Evaluation
# ======================================================

results = best_model.evaluate(valid_ds, return_dict=True)

print()

for metric, value in results.items():

    print(f"{metric}: {value:.4f}")

---

# Cell 17 – Display Metrics

In [ ]:
# ======================================================
# Final Metrics
# ======================================================

metrics = pd.DataFrame({

    "Metric": list(results.keys()),

    "Value": list(results.values())

})

metrics

---

# Cell 18 – Training Summary

In [ ]:
print("=" * 60)
print("Phase 4 Completed Successfully!")
print("=" * 60)

print()

print("Stage 1 Training Completed")

print("Fine-Tuning Completed")

print("Best Model Saved")

print("Training Logs Saved")

print("Model Ready for Evaluation")

---

# 📁 Files Generated

After training, your project directory will contain:

```text
saved_models/
│
├── best_densenet121.keras
├── stage1_model.keras
├── lung_abnormality_model.keras
└── training_log.csv
```

---

# ✅ Phase 4 Complete

By the end of this phase, you'll have:

* ✔️ Two-stage transfer learning (feature extraction + fine-tuning)
* ✔️ Automatic learning rate reduction
* ✔️ Early stopping based on validation AUC
* ✔️ Best model checkpointing
* ✔️ CSV logs for training analysis
* ✔️ Final trained model ready for inference

---

## 🚀 Next: Phase 5 – Model Evaluation & Performance Analysis

We'll evaluate the trained model with metrics and visualizations expected in an academic project:

* ROC Curve (per disease)
* AUC Score
* Precision, Recall, F1-Score
* Multi-label confusion matrices
* Threshold tuning
* Classification report
* Performance summary tables suitable for your MCA project report.

# Phase 5 – Model Evaluation & Performance Analysis

---

## Cell 1 – Import Required Libraries

In [ ]:
# ======================================================
# Imports
# ======================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    roc_curve,
    auc,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

---

# Cell 2 – Generate Predictions

In [ ]:
# ======================================================
# Generate Predictions
# ======================================================

y_true = []
y_pred_prob = []

for images, labels in valid_ds:

    predictions = best_model.predict(images, verbose=0)

    y_true.extend(labels.numpy())

    y_pred_prob.extend(predictions)

y_true = np.array(y_true)

y_pred_prob = np.array(y_pred_prob)

print("Ground Truth Shape :", y_true.shape)
print("Prediction Shape   :", y_pred_prob.shape)

Expected:

```
Ground Truth Shape : (XXXX,14)

Prediction Shape   : (XXXX,14)
```

---

# Cell 3 – Apply Threshold

In [ ]:
# ======================================================
# Binary Prediction
# ======================================================

THRESHOLD = 0.5

y_pred = (y_pred_prob >= THRESHOLD).astype(int)

print("Threshold :", THRESHOLD)

---

# Cell 4 – Overall Metrics

In [ ]:
# ======================================================
# Overall Metrics
# ======================================================

precision = precision_score(
    y_true,
    y_pred,
    average="micro",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="micro",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="micro",
    zero_division=0
)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

---

# Cell 5 – Classification Report

In [ ]:
# ======================================================
# Classification Report
# ======================================================

report = classification_report(
    y_true,
    y_pred,
    target_names=LABELS,
    zero_division=0
)

print(report)

---

# Cell 6 – Per-Class ROC Curve

In [ ]:
# ======================================================
# ROC Curves
# ======================================================

plt.figure(figsize=(12,10))

roc_scores = {}

for i, disease in enumerate(LABELS):

    fpr, tpr, _ = roc_curve(
        y_true[:, i],
        y_pred_prob[:, i]
    )

    roc_auc = auc(fpr, tpr)

    roc_scores[disease] = roc_auc

    plt.plot(
        fpr,
        tpr,
        label=f"{disease} ({roc_auc:.2f})"
    )

plt.plot([0,1],[0,1],"k--")

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend(fontsize=8)

plt.show()

---

# Cell 7 – ROC Score Table

In [ ]:
# ======================================================
# ROC Scores
# ======================================================

roc_df = pd.DataFrame({

    "Disease": list(roc_scores.keys()),

    "AUC": list(roc_scores.values())

})

roc_df = roc_df.sort_values(
    by="AUC",
    ascending=False
)

roc_df

---

# Cell 8 – Plot AUC Scores

In [ ]:
# ======================================================
# Plot AUC
# ======================================================

plt.figure(figsize=(12,6))

plt.barh(
    roc_df["Disease"],
    roc_df["AUC"]
)

plt.xlim([0,1])

plt.xlabel("AUC")

plt.title("Per Disease AUC")

plt.show()

---

# Cell 9 – Confusion Matrix (Example)

We'll display the confusion matrix for one disease (e.g., **Pneumonia**). You can change the disease name as needed.

In [ ]:
# ======================================================
# Confusion Matrix
# ======================================================

disease = "Pneumonia"

idx = LABELS.index(disease)

cm = confusion_matrix(
    y_true[:, idx],
    y_pred[:, idx]
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative","Positive"]
)

disp.plot(cmap="Blues")

plt.title(disease)

plt.show()

---

# Cell 10 – Performance Summary

In [ ]:
# ======================================================
# Performance Table
# ======================================================

summary = []

for i, disease in enumerate(LABELS):

    precision = precision_score(
        y_true[:, i],
        y_pred[:, i],
        zero_division=0
    )

    recall = recall_score(
        y_true[:, i],
        y_pred[:, i],
        zero_division=0
    )

    f1 = f1_score(
        y_true[:, i],
        y_pred[:, i],
        zero_division=0
    )

    summary.append([

        disease,

        precision,

        recall,

        f1,

        roc_scores[disease]

    ])

summary_df = pd.DataFrame(

    summary,

    columns=[
        "Disease",
        "Precision",
        "Recall",
        "F1 Score",
        "AUC"
    ]

)

summary_df

---

# Cell 11 – Save Evaluation Results

In [ ]:
# ======================================================
# Save Results
# ======================================================

summary_df.to_csv(
    "evaluation_results.csv",
    index=False
)

print("Evaluation Results Saved")

---

# Cell 12 – Display Top Performing Diseases

In [ ]:
summary_df.sort_values(
    by="AUC",
    ascending=False
).head(10)

---

# Cell 13 – Display Lowest Performing Diseases

In [ ]:
summary_df.sort_values(
    by="AUC"
).head(10)

---

# Cell 14 – Average AUC

In [ ]:
print("Average AUC :", summary_df["AUC"].mean())

---

# Cell 15 – Per-Class Threshold Tuning

The classification report and confusion matrices above all used a single
fixed threshold (0.5) for every disease. That's misleading for a multi-label
problem where prevalence varies wildly by class (e.g. Pneumonia may be ~4%
positive while Lung Opacity is much more common) — probabilities for rare
classes often never cross 0.5, so recall collapses to 0 for them even though
the model has learned real ranking signal (see the AUC chart in Cell 8).
Here we pick, per disease, the threshold that maximizes F1 on the validation
set instead of using 0.5 everywhere.

In [ ]:
# ======================================================
# Per-Class Threshold Tuning
# ======================================================

from sklearn.metrics import precision_recall_curve

optimal_thresholds = {}
threshold_rows = []

for i, disease in enumerate(LABELS):

    precisions, recalls, thresholds = precision_recall_curve(
        y_true[:, i],
        y_pred_prob[:, i]
    )

    f1_scores = np.divide(
        2 * precisions * recalls,
        precisions + recalls,
        out=np.zeros_like(precisions),
        where=(precisions + recalls) != 0
    )

    positive_count = int(y_true[:, i].sum())

    if positive_count == 0 or f1_scores[:-1].max() == 0:
        best_threshold = 0.50
        best_f1 = 0.0
    else:
        best_idx = np.argmax(f1_scores[:-1])
        best_threshold = float(thresholds[best_idx])
        best_f1 = float(f1_scores[best_idx])

    optimal_thresholds[disease] = best_threshold

    threshold_rows.append([disease, best_threshold, best_f1, positive_count])

threshold_df = pd.DataFrame(
    threshold_rows,
    columns=["Disease", "Optimal Threshold", "Best F1", "Positive Count"]
).sort_values(by="Optimal Threshold", ascending=False)

threshold_df

Classes with zero positive examples in this validation split (e.g.
Fracture) have nothing to tune against, so they fall back to the default
0.50 — that's expected, not a bug.

---

# Cell 16 – Apply Tuned Thresholds

In [ ]:
# ======================================================
# Apply Tuned Thresholds
# ======================================================

import json

threshold_array = np.array([optimal_thresholds[d] for d in LABELS])

y_pred_tuned = (y_pred_prob >= threshold_array).astype(int)

print("Fixed threshold (0.50 for every class):")
print(classification_report(y_true, y_pred, target_names=LABELS, zero_division=0))

print("\nPer-class tuned thresholds:")
print(classification_report(y_true, y_pred_tuned, target_names=LABELS, zero_division=0))

print("Subset Accuracy — fixed 0.50 :", accuracy_score(y_true, y_pred))
print("Subset Accuracy — tuned     :", accuracy_score(y_true, y_pred_tuned))

threshold_path = os.path.join(MODEL_DIR, "optimal_thresholds.json")

with open(threshold_path, "w") as f:
    json.dump(optimal_thresholds, f, indent=2)

print("\nOptimal Thresholds Saved:", threshold_path)

`optimal_thresholds.json` is what Phase 6 and Phase 7 load so the rest of
the pipeline (Grad-CAM label selection, the AI report's "significant
findings" filter) uses these per-class thresholds instead of a blanket 0.5.

---

# Cell 17 – Evaluation Complete

In [ ]:
print("="*60)
print("Phase 5 Completed Successfully!")
print("="*60)

print()

print("Classification Report Generated")

print("ROC Curves Generated")

print("Confusion Matrix Generated")

print("Evaluation CSV Saved")

print("Model Performance Verified")

---

# 📁 Files Generated

```
evaluation_results.csv
```

---

# ✅ Phase 5 Complete

At the end of this phase, you'll have:

* ✔️ Overall Precision, Recall, and F1-Score
* ✔️ Per-disease ROC curves and AUC values
* ✔️ Classification report for all 14 disease labels
* ✔️ Confusion matrix visualization for individual diseases
* ✔️ Evaluation summary saved as a CSV file

These outputs are well suited for the **Results** and **Performance Evaluation** sections of your MCA project report.

---

## 🚀 Next: Phase 6 – Explainable AI with Grad-CAM

In the next phase, we'll implement **Grad-CAM** to visualize the regions of the chest X-ray that most influenced the model's prediction. This will make your project significantly stronger by providing explainable AI, a key requirement for modern medical imaging systems.

# Phase 6 – Explainable AI with Grad-CAM

---

## Cell 1 – Import Libraries

In [ ]:
# ======================================================
# Imports
# ======================================================

import cv2
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import Model

---

# Cell 2 – Find the Backbone & Last Convolutional Layer

`best_model` wraps the DenseNet121 backbone as a **nested sub-model**
(Phase 3 builds it via `x = base_model(x, training=False)`), so the
convolutional layers live *inside* that nested model — they are not direct
layers of `best_model`. We first find the nested backbone, then search
inside it.

In [ ]:
# ======================================================
# Locate Backbone & Last Convolution Layer
# ======================================================

BACKBONE_NAME = None

for layer in best_model.layers:

    if isinstance(layer, tf.keras.Model):

        BACKBONE_NAME = layer.name

        break

if BACKBONE_NAME is None:
    raise ValueError("Could not find a nested backbone model inside best_model.")

backbone = best_model.get_layer(BACKBONE_NAME)

backbone_layer_names = [l.name for l in backbone.layers]

if "conv5_block16_concat" in backbone_layer_names:

    LAST_CONV_LAYER = "conv5_block16_concat"

else:

    LAST_CONV_LAYER = None

    for layer in reversed(backbone.layers):

        if isinstance(layer, tf.keras.layers.Conv2D):

            LAST_CONV_LAYER = layer.name

            break

print("Backbone:", BACKBONE_NAME)
print("Last Conv Layer:", LAST_CONV_LAYER)

Expected output:

```
Backbone: densenet121
Last Conv Layer: conv5_block16_concat
```

---

# Cell 3 – Create Grad-CAM Function

Because the backbone is nested, we can't build a single `grad_model` from
`model.get_layer(last_conv_layer_name)` the way a flat (non-nested)
architecture would allow — that only searches `model`'s direct children.
Instead we rebuild the forward pass in two pieces: **(1)** backbone input →
last conv layer output, and **(2)** that output fed through whichever
layers come *after* the backbone in `model.layers` (GlobalAveragePooling2D,
BatchNormalization, Dense, Dropout, the sigmoid output layer). Gradients are
then taken across both pieces together inside one `GradientTape`.

In [ ]:
# ======================================================
# Grad-CAM Function (nested-model safe)
# ======================================================

def make_gradcam_heatmap(
    image,
    model,
    last_conv_layer_name,
    pred_index=None
):

    backbone = None
    backbone_index = None

    for i, layer in enumerate(model.layers):

        if isinstance(layer, tf.keras.Model):

            backbone = layer

            backbone_index = i

            break

    if backbone is None:
        raise ValueError("No nested backbone model found inside `model`.")

    last_conv_layer = backbone.get_layer(last_conv_layer_name)

    last_conv_layer_model = Model(
        backbone.input,
        last_conv_layer.output
    )

    classifier_input = tf.keras.Input(
        shape=last_conv_layer.output.shape[1:]
    )

    x = classifier_input

    for layer in model.layers[backbone_index + 1:]:

        x = layer(x)

    classifier_model = Model(classifier_input, x)

    with tf.GradientTape() as tape:

        conv_outputs = last_conv_layer_model(image)

        tape.watch(conv_outputs)

        predictions = classifier_model(conv_outputs)

        if pred_index is None:
            pred_index = tf.argmax(predictions[0])

        class_channel = predictions[:, pred_index]

    grads = tape.gradient(
        class_channel,
        conv_outputs
    )

    pooled_grads = tf.reduce_mean(
        grads,
        axis=(0,1,2)
    )

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]

    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(
        heatmap,
        0
    ) / tf.math.reduce_max(heatmap)

    return heatmap.numpy()

---

# Cell 4 – Load an Image

In [ ]:
# ======================================================
# Select Image
# ======================================================

sample_path = valid_df.iloc[0]["image_path"]

print(sample_path)

---

# Cell 5 – Preprocess Image

In [ ]:
# ======================================================
# Preprocess
# ======================================================

img = tf.keras.preprocessing.image.load_img(
    sample_path,
    target_size=(224,224)
)

img_array = tf.keras.preprocessing.image.img_to_array(img)

img_array = img_array / 255.0

img_array = np.expand_dims(
    img_array,
    axis=0
)

---

# Cell 6 – Predict Diseases

In [ ]:
# ======================================================
# Prediction
# ======================================================

prediction = best_model.predict(img_array)

prediction = prediction[0]

prediction_df = pd.DataFrame({

    "Disease": LABELS,

    "Probability": prediction

})

prediction_df = prediction_df.sort_values(
    by="Probability",
    ascending=False
)

prediction_df.head(10)

---

# Cell 7 – Generate Heatmap

In [ ]:
# ======================================================
# Grad-CAM
# ======================================================

predicted_class = np.argmax(prediction)

heatmap = make_gradcam_heatmap(

    img_array,

    best_model,

    LAST_CONV_LAYER,

    pred_index=predicted_class

)

plt.imshow(heatmap)

plt.colorbar()

plt.title("Grad-CAM Heatmap")

plt.show()

---

# Cell 8 – Overlay Heatmap

In [ ]:
# ======================================================
# Overlay
# ======================================================

original = cv2.imread(sample_path)

original = cv2.cvtColor(
    original,
    cv2.COLOR_BGR2RGB
)

original = cv2.resize(
    original,
    (224,224)
)

heatmap = cv2.resize(
    heatmap,
    (224,224)
)

heatmap = np.uint8(255 * heatmap)

heatmap = cv2.applyColorMap(
    heatmap,
    cv2.COLORMAP_JET
)

superimposed = cv2.addWeighted(

    original,

    0.6,

    heatmap,

    0.4,

    0

)

---

# Cell 9 – Display Results

In [ ]:
# ======================================================
# Visualization
# ======================================================

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)

plt.imshow(original)

plt.title("Original")

plt.axis("off")

plt.subplot(1,3,2)

plt.imshow(heatmap)

plt.title("Heatmap")

plt.axis("off")

plt.subplot(1,3,3)

plt.imshow(superimposed)

plt.title("Grad-CAM")

plt.axis("off")

plt.show()

---

# Cell 10 – Display Predicted Diseases

Uses the per-class thresholds tuned in Phase 5 (`optimal_thresholds.json`)
instead of a single 0.50 cutoff for every disease — falls back to 0.50 for
any disease missing from the file (e.g. if Phase 5 hasn't been run yet).

In [ ]:
# ======================================================
# Predictions
# ======================================================

import json

try:
    with open("saved_models/optimal_thresholds.json", "r") as f:
        thresholds = json.load(f)
except FileNotFoundError:
    thresholds = {}

print("Predicted Diseases\n")

for disease, prob in zip(

    LABELS,

    prediction

):

    disease_threshold = thresholds.get(disease, 0.50)

    if prob >= disease_threshold:

        print(f"✓ {disease} : {prob:.2%} (threshold {disease_threshold:.0%})")

---

# Cell 11 – Save Image

In [ ]:
# ======================================================
# Save
# ======================================================

plt.imsave(

    "gradcam_result.png",

    superimposed

)

print("Saved Successfully")

---

# Cell 12 – Test Multiple Images

In [ ]:
# ======================================================
# Random Samples
# ======================================================

sample_images = valid_df.sample(
    5,
    random_state=42
)

sample_images[["image_path"]]

---

# Cell 13 – Phase Complete

In [ ]:
print("="*60)

print("Phase 6 Completed Successfully!")

print("="*60)

print()

print("Grad-CAM Generated")

print("Explainable AI Enabled")

print("Medical Visualization Ready")

print("Image Saved")

---

# 📁 Output

```
gradcam_result.png
```

---

# ✅ Phase 6 Complete

You now have:

* ✔️ Explainable AI using Grad-CAM
* ✔️ Heatmap visualization
* ✔️ Overlay on the original chest X-ray
* ✔️ Disease probability table
* ✔️ Exportable Grad-CAM image

This is a significant enhancement for your project because it provides clinicians with a visual explanation of the model's predictions.

---

## ✅ Nested-Model Issue Resolved

Earlier drafts of this phase called `model.get_layer(last_conv_layer_name)`
directly, which fails whenever the backbone is nested (as it is here, from
Phase 3's `x = base_model(x, training=False)`). Cells 2–3 above now locate
the nested backbone first and rebuild the forward pass in two pieces
(backbone → last conv layer, then the remaining classifier layers), so
Grad-CAM works correctly against the real nested architecture — no changes
to Phase 3 required. You can generate Grad-CAM for **any of the 14
diseases** by passing a different `pred_index` to `make_gradcam_heatmap`.

---

## 🚀 Next: Phase 7 – Prediction & AI Report Generation

In the next phase, we'll build the inference pipeline that:

* Accepts any uploaded chest X-ray.
* Predicts probabilities for all 14 disease labels.
* Generates a structured AI medical report.
* Displays the Grad-CAM overlay.
* Produces a PDF-style report suitable for your web application.

# Phase 7 – Chest X-ray Prediction & AI Report Generation

---

## Cell 1 – Import Libraries

In [ ]:
# ======================================================
# Imports
# ======================================================

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime
import json

---

# Cell 2 – Configuration

Loads the per-class thresholds tuned in Phase 5 (`optimal_thresholds.json`)
instead of one flat cutoff for all 14 diseases — a class with a tuned
threshold of, say, 0.30 will now correctly show up as a finding at 35%
confidence instead of being silently dropped by a blanket 0.50. Falls back
to `DEFAULT_THRESHOLD` for any disease missing from the file.

In [ ]:
# ======================================================
# Configuration
# ======================================================

IMG_SIZE = 224

DEFAULT_THRESHOLD = 0.50

THRESHOLDS_PATH = "saved_models/optimal_thresholds.json"

try:
    with open(THRESHOLDS_PATH, "r") as f:
        CONFIDENCE_THRESHOLDS = json.load(f)
except FileNotFoundError:
    CONFIDENCE_THRESHOLDS = {}

---

# Cell 3 – Load Trained Model

In [ ]:
# ======================================================
# Load Model
# ======================================================

model = tf.keras.models.load_model(
    "saved_models/best_densenet121.keras"
)

print("Model Loaded Successfully")

---

# Cell 4 – Image Preprocessing

In [ ]:
# ======================================================
# Image Preprocessing
# ======================================================

def preprocess_image(image_path):

    image = Image.open(image_path).convert("RGB")

    image = image.resize((IMG_SIZE, IMG_SIZE))

    image_array = np.array(image).astype(np.float32)

    image_array /= 255.0

    image_array = np.expand_dims(image_array, axis=0)

    return image_array, image

---

# Cell 5 – Severity Estimation

In [ ]:
# ======================================================
# Severity
# ======================================================

def estimate_severity(probability):

    if probability >= 0.90:
        return "Critical"

    elif probability >= 0.75:
        return "High"

    elif probability >= 0.60:
        return "Moderate"

    elif probability >= 0.50:
        return "Low"

    else:
        return "Normal"

---

# Cell 6 – Prediction Function

In [ ]:
# ======================================================
# Prediction
# ======================================================

def predict_xray(image_path):

    image_array, original = preprocess_image(image_path)

    predictions = model.predict(
        image_array,
        verbose=0
    )[0]

    results = []

    for disease, probability in zip(LABELS, predictions):

        severity = estimate_severity(probability)

        results.append({

            "Disease": disease,

            "Probability": float(probability),

            "Severity": severity

        })

    results = pd.DataFrame(results)

    results = results.sort_values(

        by="Probability",

        ascending=False

    )

    return original, results

---

# Cell 7 – Select Test Image

In [ ]:
# ======================================================
# Test Image
# ======================================================

IMAGE_PATH = valid_df.iloc[10]["image_path"]

print(IMAGE_PATH)

You can later replace it with:

In [ ]:
IMAGE_PATH = "/content/my_xray.png"

---

# Cell 8 – Run Prediction

In [ ]:
# ======================================================
# Predict
# ======================================================

original, prediction_df = predict_xray(IMAGE_PATH)

prediction_df.head(10)

---

# Cell 9 – Display X-ray

In [ ]:
# ======================================================
# Display Image
# ======================================================

plt.figure(figsize=(6,6))

plt.imshow(original)

plt.axis("off")

plt.title("Chest X-ray")

plt.show()

---

# Cell 10 – Display Significant Findings

In [ ]:
# ======================================================
# Significant Findings
# ======================================================

findings = prediction_df[
    prediction_df.apply(
        lambda row: row["Probability"] >= CONFIDENCE_THRESHOLDS.get(
            row["Disease"], DEFAULT_THRESHOLD
        ),
        axis=1
    )
]

findings

---

# Cell 11 – AI Medical Report

In [ ]:
# ======================================================
# Report
# ======================================================

print("="*60)

print("AI RADIOLOGY REPORT")

print("="*60)

print()

print("Report Generated :", datetime.now())

print()

if len(findings) == 0:

    print("No significant lung abnormality detected.")

else:

    print("Detected Findings:\n")

    for _, row in findings.iterrows():

        print(

            f"• {row['Disease']}"

            f" | Confidence: {row['Probability']:.2%}"

            f" | Severity: {row['Severity']}"

        )

print()

print("="*60)

print("This report is AI-assisted and intended")

print("to support clinical decision-making.")

print("="*60)

---

# Cell 12 – Save Report

In [ ]:
# ======================================================
# Save Report
# ======================================================

prediction_df.to_csv(

    "prediction_report.csv",

    index=False

)

print("Report Saved Successfully")

---

# Cell 13 – Create Report Dictionary

Useful for APIs and your Next.js frontend.

In [ ]:
# ======================================================
# API Output
# ======================================================

report = {

    "generated_at": str(datetime.now()),

    "top_prediction": prediction_df.iloc[0]["Disease"],

    "confidence": float(

        prediction_df.iloc[0]["Probability"]

    ),

    "findings": findings.to_dict(

        orient="records"

    )

}

report

---

# Cell 14 – Final Output

In [ ]:
print("="*60)

print("Phase 7 Completed Successfully!")

print("="*60)

print()

print("Prediction Pipeline Ready")

print("AI Report Ready")

print("Frontend Integration Ready")

print("CSV Report Generated")

---

# Example Output

```
============================================================

AI RADIOLOGY REPORT

============================================================

Detected Findings

• Lung Opacity
Confidence : 96.24%
Severity : Critical

• Pleural Effusion
Confidence : 81.33%
Severity : High

• Cardiomegaly
Confidence : 73.42%
Severity : Moderate
```

---

# Files Generated

```
prediction_report.csv
```

---

# Phase 7 Complete

Your system can now:

* ✅ Load any chest X-ray image
* ✅ Predict all 14 CheXpert disease labels
* ✅ Assign confidence scores
* ✅ Estimate severity levels
* ✅ Generate a structured AI-assisted radiology report
* ✅ Export predictions to CSV
* ✅ Produce a JSON-compatible output for frontend/API integration

---

## 🚀 Phase 8 – Hugging Face Deployment & Gradio Interface

In the final phase, we'll deploy the trained model as a **Gradio application on Hugging Face Spaces**. You'll get:

* A professional drag-and-drop web interface
* Chest X-ray upload
* AI predictions
* Grad-CAM visualization
* AI-generated report
* Public API endpoint
* Integration guide for your **Next.js** frontend

After Phase 8, your MCA project will be fully deployable and ready for demonstration.

# Phase 8 – Deploy to Hugging Face Spaces with Gradio

Running the cells below writes out the Gradio Space files (`labels.py`, `utils.py`, `predict.py`, `app.py`, `requirements.txt`, `README.md`) into the current working directory, ready to upload to a Hugging Face Space alongside `best_densenet121.keras`.

## Cell 1 – Install Gradio

In [ ]:
!pip install gradio opencv-python pillow

## File: `labels.py`

In [ ]:
%%writefile labels.py
LABELS = [
    "No Finding",
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Opacity",
    "Lung Lesion",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices"
]

## File: `utils.py`

In [ ]:
%%writefile utils.py
from PIL import Image
import numpy as np

IMG_SIZE = 224

def preprocess(image):

    image = image.convert("RGB")

    image = image.resize((IMG_SIZE, IMG_SIZE))

    image = np.array(image).astype(np.float32)

    image /= 255.0

    image = np.expand_dims(image, axis=0)

    return image

## File: `predict.py`

In [ ]:
%%writefile predict.py
import tensorflow as tf
import pandas as pd

from labels import LABELS
from utils import preprocess

model = tf.keras.models.load_model(
    "best_densenet121.keras"
)

def predict(image):

    image_array = preprocess(image)

    pred = model.predict(
        image_array,
        verbose=0
    )[0]

    df = pd.DataFrame({

        "Disease": LABELS,

        "Confidence": pred

    })

    df = df.sort_values(

        by="Confidence",

        ascending=False

    )

    return df

## File: `app.py`

In [ ]:
%%writefile app.py
import gradio as gr
from predict import predict

def inference(image):

    result = predict(image)

    report = ""

    for _, row in result.iterrows():

        if row["Confidence"] > 0.50:

            report += (
                f"{row['Disease']} "
                f"({row['Confidence']:.2%})\n"
            )

    if report == "":

        report = "No significant abnormalities detected."

    return result, report


demo = gr.Interface(

    fn=inference,

    inputs=gr.Image(type="pil"),

    outputs=[

        gr.Dataframe(),

        gr.Textbox()

    ],

    title="AI Lung Abnormality Detection",

    description="""
Upload a Chest X-ray to detect
possible lung abnormalities.
""",

    allow_flagging="never"

)

demo.launch()

## File: `requirements.txt`

In [ ]:
%%writefile requirements.txt
tensorflow
gradio
numpy
pandas
opencv-python-headless
Pillow
matplotlib

## File: `README.md`

In [ ]:
%%writefile README.md
# AI Lung Abnormality Detection

This application detects multiple lung abnormalities from Chest X-rays using DenseNet121 trained on the CheXpert dataset.

## Features

- Multi-label disease prediction
- Confidence scores
- AI-assisted radiology report
- Chest X-ray upload
- Ready for Hugging Face deployment

## Dataset

CheXpert

## Model

DenseNet121

## Framework

TensorFlow

## Frontend

Gradio

# Upload to Hugging Face

Create a new **Gradio Space** and upload:

```text
app.py
requirements.txt
README.md
best_densenet121.keras
predict.py
utils.py
labels.py
```

Hugging Face will automatically install the dependencies and launch your application.

---

# Integration with Next.js

Once deployed, you can call the Hugging Face Space from your Next.js app:

```ts
const response = await fetch("YOUR_SPACE_URL/run/predict", {
  method: "POST",
  headers: {
    "Content-Type": "application/json",
  },
  body: JSON.stringify({
    data: [imageBase64],
  }),
});

const result = await response.json();
```

---

# Optional Improvements

For a more polished application, you can extend it with:

* Patient information form (name, age, gender, ID)
* Upload history stored in PostgreSQL
* PDF report generation
* Interactive Grad-CAM visualization
* Severity indicators (Low, Moderate, High, Critical)
* Doctor dashboard with patient history
* REST API using FastAPI
* Authentication with Clerk or NextAuth
* Docker deployment for self-hosting

---

# 🎓 MCA Project Outcome

By completing all phases, you have built:

* **AI Model:** DenseNet121 trained on CheXpert for multi-label lung abnormality detection.
* **Explainable AI:** Grad-CAM visualization for highlighting influential regions in X-rays.
* **Inference Pipeline:** Upload, preprocess, predict, and generate AI-assisted reports.
* **Evaluation:** Precision, Recall, F1-score, ROC-AUC, and confusion matrices.
* **Deployment:** A Gradio-based web application deployable on Hugging Face Spaces.
* **Integration Ready:** The deployed model can be connected to your Next.js frontend through an API.

### A recommendation before deployment

Before uploading to Hugging Face, consider refactoring the code into reusable modules (`model.py`, `predict.py`, `gradcam.py`, `report.py`) and adding proper error handling and logging. This will make the application easier to maintain, test, and extend when you integrate it into your full-stack MCA project.